In [4]:
import os
import time
import shutil
import re
from datetime import datetime
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
from geopy.geocoders import Nominatim

# 住所取得のための初期設定
geolocator = Nominatim(user_agent="exif_photo_sorter")

def sanitize_filename(name):
    """フォルダ名として使用できない記号をアンダースコアに置換する"""
    return re.sub(r'[\\/:*?"<>|]+', '_', name)

def get_location_name(lat, lon):
    """緯度経度から詳細な住所と施設名を取得する"""
    try:
        time.sleep(1)
        location = geolocator.reverse(f"{lat}, {lon}", language='ja', addressdetails=True)
        if location:
            addr = location.raw.get('address', {})
            
            # 1. 施設名（POI）の抽出
            # OpenStreetMapで施設名が入りやすいキーを順番に探す
            poi_keys = ['amenity', 'building', 'shop', 'tourism', 'leisure', 'historic', 'railway', 'station', 'office']
            poi_name = ""
            for key in poi_keys:
                if key in addr:
                    poi_name = addr[key]
                    break
            
            # 2. 詳細住所の組み立て
            pref = addr.get('province', '') or addr.get('state', '')
            city = addr.get('city', '') or addr.get('town', '') or addr.get('village', '') or addr.get('county', '')
            suburb = addr.get('suburb', '')               # 区、町域
            neighbourhood = addr.get('neighbourhood', '') # 丁目など
            quarter = addr.get('quarter', '')             # 街区、番地など
            road = addr.get('road', '')                   # 通り名、道路名

            # 基本の住所文字列を作成
            detail_address = f"{pref}{city}{suburb}{neighbourhood}{quarter}"
            
            # 番地等がなく、道路名だけが取得できた場合は追加する
            if road and not neighbourhood and not quarter:
                detail_address += f"_{road}"

            # 取得に失敗した場合の予備処理（カンマ区切りの最初の部分を使用）
            if not detail_address:
                detail_address = location.address.split(',')[0].strip()

            # 3. 施設名があれば住所の末尾に追加
            if poi_name:
                full_name = f"{detail_address}_{poi_name}"
            else:
                full_name = detail_address
            
            # フォルダ名として安全な文字列に変換して返す
            return sanitize_filename(full_name)
            
        return "場所不明"
    except Exception:
        return "場所取得エラー"

def get_exif_data(image_path):
    try:
        with Image.open(image_path) as img:
            exif = img._getexif()
            if not exif:
                return None
            return {TAGS.get(key, key): val for key, val in exif.items()}
    except Exception:
        return None

def get_gps_info(exif_data):
    if 'GPSInfo' not in exif_data:
        return None
    gps_info = {}
    for key, val in exif_data['GPSInfo'].items():
        decode_name = GPSTAGS.get(key, key)
        gps_info[decode_name] = val
    return gps_info

def get_shooting_datetime(exif_data):
    if not exif_data:
        return None
    date_str = exif_data.get('DateTimeOriginal') or exif_data.get('DateTime')
    if date_str:
        try:
            return datetime.strptime(date_str, "%Y:%m:%d %H:%M:%S")
        except ValueError:
            pass
    return None

def convert_to_decimal(dms_value, ref):
    try:
        degrees = float(dms_value[0])
        minutes = float(dms_value[1])
        seconds = float(dms_value[2])
        decimal = degrees + (minutes / 60.0) + (seconds / 3600.0)
        if ref in ['S', 'W']:
            decimal = -decimal
        return decimal
    except Exception:
        return None

def get_coordinates(gps_info):
    if not gps_info:
        return None, None
    lat, lon = None, None
    if 'GPSLatitude' in gps_info and 'GPSLatitudeRef' in gps_info:
        lat = convert_to_decimal(gps_info['GPSLatitude'], gps_info['GPSLatitudeRef'])
    if 'GPSLongitude' in gps_info and 'GPSLongitudeRef' in gps_info:
        lon = convert_to_decimal(gps_info['GPSLongitude'], gps_info['GPSLongitudeRef'])
    return lat, lon

def classify_photos_by_time_gap(target_folder, gap_minutes=10):
    output_base_dir = os.path.join(target_folder, "_現場別_自動仕分け")
    os.makedirs(output_base_dir, exist_ok=True)

    print(f"[{target_folder}] の解析を開始します...")
    print(f"※前の写真から {gap_minutes} 分以上空いた場合、別の現場としてフォルダを分けます。\n")

    photos_with_time = []
    photos_without_time = []

    for filename in os.listdir(target_folder):
        if not filename.lower().endswith(('.jpg', '.jpeg')):
            continue

        filepath = os.path.join(target_folder, filename)
        exif_data = get_exif_data(filepath)
        
        dt = get_shooting_datetime(exif_data)
        lat, lon = None, None
        
        if exif_data:
            gps_info = get_gps_info(exif_data)
            lat, lon = get_coordinates(gps_info)

        photo_info = {
            'filename': filename,
            'filepath': filepath,
            'dt': dt,
            'lat': lat,
            'lon': lon
        }

        if dt:
            photos_with_time.append(photo_info)
        else:
            photos_without_time.append(photo_info)

    photos_with_time.sort(key=lambda x: x['dt'])

    groups = []
    current_group = []
    last_time = None

    for photo in photos_with_time:
        if last_time is None:
            current_group.append(photo)
        else:
            time_diff = photo['dt'] - last_time
            if time_diff.total_seconds() >= gap_minutes * 60:
                groups.append(current_group)
                current_group = [photo]
            else:
                current_group.append(photo)
        last_time = photo['dt']
    
    if current_group:
        groups.append(current_group)

    print("=== 分類結果とフォルダ作成 ===")
    
    for group in groups:
        first_photo_dt = group[0]['dt']
        date_str = first_photo_dt.strftime("%Y-%m-%d")
        time_str = first_photo_dt.strftime("%H%M")
        
        location_name = "場所不明"
        for photo in group:
            if photo['lat'] is not None and photo['lon'] is not None:
                location_name = get_location_name(photo['lat'], photo['lon'])
                break
                
        folder_name = f"{date_str}_{time_str}_{location_name}"
        group_dir = os.path.join(output_base_dir, folder_name)
        os.makedirs(group_dir, exist_ok=True)
        
        print(f"\n📁 現場フォルダ: {folder_name} ({len(group)}枚)")
        
        for photo in group:
            shutil.copy2(photo['filepath'], os.path.join(group_dir, photo['filename']))
            print(f"    - コピー完了: {photo['filename']}")

    if photos_without_time:
        no_time_dir = os.path.join(output_base_dir, "日時情報なし")
        os.makedirs(no_time_dir, exist_ok=True)
        print(f"\n⚠️ 日時情報がないファイル ({len(photos_without_time)}枚) を専用フォルダへコピーします:")
        for photo in photos_without_time:
            shutil.copy2(photo['filepath'], os.path.join(no_time_dir, photo['filename']))
            print(f"    - {photo['filename']}")

    print(f"\n✅ すべての処理が完了しました！\n{output_base_dir}")

# ==========================================
# 実行部分
# ==========================================
FOLDER_PATH = r"C:\現地写真"

if os.path.exists(FOLDER_PATH):
    classify_photos_by_time_gap(FOLDER_PATH, gap_minutes=10)
else:
    print(f"エラー: フォルダ '{FOLDER_PATH}' が見つかりません。")

[C:\現地写真] の解析を開始します...
※前の写真から 10 分以上空いた場合、別の現場としてフォルダを分けます。

=== 分類結果とフォルダ作成 ===

📁 現場フォルダ: 2026-07-01_0835_滋賀県大津市馬場二丁目竜が丘_滋賀国道事務所 (4枚)
    - コピー完了: 20260701_083551.jpg
    - コピー完了: 20260701_083600.jpg
    - コピー完了: 20260701_083622.jpg
    - コピー完了: 20260701_083638.jpg

📁 現場フォルダ: 2026-07-01_0854_滋賀県大津市馬場二丁目竜が丘_リカーマウンテン (298枚)
    - コピー完了: 20260701_085444.jpg
    - コピー完了: 20260701_085446.jpg
    - コピー完了: IMG_4076.jpg
    - コピー完了: IMG_4077.jpg
    - コピー完了: IMG_4078.jpg
    - コピー完了: IMG_4079.jpg
    - コピー完了: IMG_4080.jpg
    - コピー完了: IMG_4081.jpg
    - コピー完了: IMG_4082.jpg
    - コピー完了: IMG_4083.jpg
    - コピー完了: IMG_4084.jpg
    - コピー完了: IMG_4085.jpg
    - コピー完了: IMG_4086.jpg
    - コピー完了: IMG_4087.jpg
    - コピー完了: IMG_4088.jpg
    - コピー完了: IMG_4089.jpg
    - コピー完了: IMG_4090.jpg
    - コピー完了: IMG_4091.jpg
    - コピー完了: IMG_4092.jpg
    - コピー完了: IMG_4093.jpg
    - コピー完了: IMG_4094.jpg
    - コピー完了: IMG_4095.jpg
    - コピー完了: IMG_4096.jpg
    - コピー完了: IMG_4097.jpg
    - コピー完了: IMG_4098.jpg
    - コピー